# 📐 Monocular Depth Pipeline — Relative + Metric

```
Image → Relative Model → Relative Map → Metric Model → Scale Alignment → Metric Map (metres)
```

| Stage | Model choices |
|-------|---------------|
| **Relative** | Depth Anything V2 (S/B/L), Marigold |
| **Metric**   | Depth Pro (Apple), UniDepthV2, DAv2 Metric |

> **Setup:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── Cell 1: Environment check ─────────────────────────────────────────────────
import sys, torch

IN_COLAB = 'google.colab' in sys.modules

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Environment : {"Google Colab" if IN_COLAB else "Local"}')
print(f'Device      : {DEVICE}')
if DEVICE == 'cpu':
    print('  ⚠️  CPU only — will be slow. Enable T4 GPU in Colab for best results.')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
# Run once. Takes ~3-4 min.
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Core
pip('torch>=2.1.0', 'torchvision>=0.16.0')
pip('transformers>=4.40.0', 'huggingface_hub>=0.22.0', 'accelerate')
pip('diffusers>=0.27.0')                   # for Marigold
pip('opencv-python-headless>=4.9.0')
pip('Pillow>=10.0.0', 'numpy>=1.24.0', 'matplotlib>=3.8.0', 'tqdm')

# UniDepthV2 (installs from GitHub)
try:
    import unidepth
    print('  UniDepth already installed')
except ImportError:
    print('  Installing UniDepthV2 from GitHub …')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/lpiccinelli-eth/UniDepth.git'
    ])

print('\n✅ All packages installed')

In [ ]:
# ── Cell 3: Upload monocular_pipeline.py ─────────────────────────────────────
import sys, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
TARGET   = 'monocular_pipeline.py'

if IN_COLAB:
    from google.colab import files
    print('Upload monocular_pipeline.py …')
    uploaded = files.upload()
    py_files = [f for f in uploaded if 'monocular_pipeline' in f and f.endswith('.py')]
    if not py_files:
        raise FileNotFoundError('Please upload monocular_pipeline.py')
    if py_files[0] != TARGET:
        shutil.copy(py_files[0], TARGET)
        print(f'  (renamed to {TARGET})')
    print(f'✅ {TARGET} ready')
else:
    if not Path(TARGET).exists():
        raise FileNotFoundError(f'{TARGET} not found in current folder.')
    print(f'✅ {TARGET} found')

In [ ]:
# ── Cell 4: Upload dataset images ────────────────────────────────────────────
import shutil, sys
from pathlib import Path

DATASET_DIR = Path('dataset')
DATASET_DIR.mkdir(exist_ok=True)
IMG_EXTS    = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}
IN_COLAB    = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files
    print('Select your images (jpg/png/etc) — multiple files OK:')
    uploaded = files.upload()
    moved = []
    for fname in uploaded:
        if Path(fname).suffix.lower() in IMG_EXTS:
            shutil.move(fname, DATASET_DIR / fname)
            moved.append(fname)
    print(f'\n✅ {len(moved)} image(s) ready in dataset/')
else:
    imgs = [f for f in DATASET_DIR.glob('*') if f.suffix.lower() in IMG_EXTS]
    print(f'✅ {len(imgs)} image(s) found in dataset/')

In [ ]:
# ── Cell 5: Choose models and run ────────────────────────────────────────────
#
# RELATIVE MODEL OPTIONS (--relative):
#   depth-anything-v2   ← fastest, good quality           [recommended to start]
#   depth-anything-v2b  ← Base model
#   depth-anything-v2l  ← Large model, best relative quality
#   marigold            ← diffusion-based, slow but detailed
#
# METRIC MODEL OPTIONS (--metric):
#   depth-pro           ← Apple DepthPro, best metric, ~1 GB  [recommended]
#   unidepth            ← UniDepthV2, fast and accurate
#   dav2-metric         ← Depth Anything V2 Metric (indoor)

import sys
sys.argv = [
    'monocular_pipeline.py',
    '--input',    'dataset',
    '--output',   'output',
    '--relative', 'depth-anything-v2',   # ← change me
    '--metric',   'depth-pro',           # ← change me
    '--colormap', 'inferno',
    # '--device', 'mps',                 # ← uncomment on Mac M1/M2/M3
]

exec(open('monocular_pipeline.py').read())

In [ ]:
# ── Cell 6: Preview composite outputs ────────────────────────────────────────
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

composites = sorted(Path('output').glob('*_composite.jpg'))
if not composites:
    print('No composites found — did Cell 5 complete?')
else:
    for comp in composites:
        img = mpimg.imread(str(comp))
        fig, ax = plt.subplots(figsize=(22, 5))
        ax.imshow(img)
        ax.set_title(comp.stem, fontsize=11)
        ax.axis('off')
        plt.tight_layout()
        plt.show()

In [ ]:
# ── Cell 7: Show JSON results ─────────────────────────────────────────────────
import json
from pathlib import Path

json_path = Path('output/results.json')
if not json_path.exists():
    print('results.json not found — run Cell 5 first.')
else:
    data = json.loads(json_path.read_text())
    for r in data:
        print(f"\n📷 {r['file']}  ({r['image_size'][0]}x{r['image_size'][1]})")
        print(f"   Relative model  : {r['relative_model']}  range={r['relative_range']}")
        print(f"   Metric model    : {r['metric_model']}")
        print(f"   Metric depth    : mean={r['metric_mean_m']} m  "
              f"median={r['metric_median_m']} m  std={r['metric_std_m']} m")
        print(f"   Metric range    : {r['metric_range_m']} m")
        print(f"   Aligned range   : {r['aligned_range_m']} m")

In [ ]:
# ── Cell 8: Compare metric depth between two model combos ────────────────────
# Run this after running Cell 5 twice with different model choices.
# Loads the two saved .npy metric depth maps and shows them side by side.

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

npys = sorted(Path('output').glob('*_metric.npy'))
if not npys:
    print('No .npy files found yet.')
else:
    for npy in npys:
        d = np.load(npy)
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        im = axes[0].imshow(d, cmap='plasma')
        axes[0].set_title(f'{npy.stem}  (metric, metres)')
        axes[0].axis('off')
        plt.colorbar(im, ax=axes[0], label='metres')

        axes[1].hist(d.flatten(), bins=100, color='steelblue', edgecolor='none')
        axes[1].set_xlabel('Depth (m)')
        axes[1].set_ylabel('Pixel count')
        axes[1].set_title('Depth distribution')
        plt.tight_layout()
        plt.show()

In [ ]:
# ── Cell 9: Download all outputs ─────────────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import shutil
    shutil.make_archive('monocular_output', 'zip', 'output')
    from google.colab import files
    files.download('monocular_output.zip')
    print('✅ Download started')
else:
    print('Local run — results are in output/')